# Agentic Metric Evaluation

An LLM agent evaluates model outputs in MongoDB by *reasoning about what each field is*,
then calling the matching metric tool over MCP and saving the verdict to the
`agentic_evaluations` collection.

This mirrors `RecordLinkage.ipynb` (same agent loop, same backend switch); only the
MCP server and the system prompt change.

**Flow per output:** `list_outputs` -> `get_task_output` -> classify each field ->
call metric tool(s) -> `save_evaluation`.

Start the server first: `./run_metric_mcp.sh`, then paste its URL into the config cell.

In [1]:
import csv
import json
import re
import time
import hashlib
import subprocess
import os
from dotenv import load_dotenv
from datetime import datetime
from pathlib import Path

import weave
from openai import OpenAI
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

## Config

In [2]:
BACKEND = "nim"  # "playground" or "nim"
MCP_URL = "http://127.0.0.1:53769/mcp"  # replace after you run ./run_metric_mcp.sh
MAX_STEPS = 12

BACKENDS = {
    "playground": {
        "base_url": "https://aiapi-prod.stanford.edu/v1",
        "api_key_file": "api_darc.txt",
        "model": "gpt-5-mini",
        "completion_kwargs": {
            "reasoning_effort": "high",
        },
    },
    "nim": {
        "base_url": "http://yen-gpu4:8000/v1",
        "api_key_file": None,
        "model": "google/gemma-4-31b-it",
        "completion_kwargs": {
            "max_tokens": 10000,
            "temperature": 0.1,
            "parallel_tool_calls": False,
            "extra_body": {
                "chat_template_kwargs": {"enable_thinking": True},
            },
        },
    },
}

cfg = BACKENDS[BACKEND]

if cfg["api_key_file"]:
    with open(cfg["api_key_file"]) as f:
        api_key = f.read().strip()
else:
    api_key = "not-used"

llm_client = OpenAI(base_url=cfg["base_url"], api_key=api_key)
MODEL = cfg["model"]
COMPLETION_KWARGS = cfg["completion_kwargs"]

print(f"Backend: {BACKEND}  |  Model: {MODEL}  |  MCP: {MCP_URL}")

Backend: nim  |  Model: google/gemma-4-31b-it  |  MCP: http://127.0.0.1:53769/mcp


In [3]:
load_dotenv()

# ── Weave observability ──────────────────────────────────────────────
# W&B auth: run `wandb login` once, or set the WANDB_API_KEY env var. Never hardcode the key.
WEAVE_PROJECT = "darc/metric-eval-agent"
PROMPT_NAME = "baseline_v1"  # bump this label whenever METRIC_EVAL_SYSTEM changes

weave.init(WEAVE_PROJECT)
print(f"Weave project: {WEAVE_PROJECT}")

/home/users/ltdarc/zfs/projects/students/ltdarc-usf-intern-2025/venv/lib/python3.10/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
weave: Logged in as Weights & Biases user: ltdarc.
weave: View Weave data at https://wandb.ai/darc/metric-eval-agent/weave


Weave project: darc/metric-eval-agent


## Helpers

In [4]:
def log(section, message=""):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"\n[{ts}] {section}")
    if message:
        print(message)

def pretty(obj):
    return json.dumps(obj, indent=2, default=str)

def preview_text(text, max_chars=1200):
    if text is None:
        return ""
    text = str(text)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + f"\n... [truncated {len(text) - max_chars} chars]"

def usage_to_dict(response):
    usage = getattr(response, "usage", None)
    if usage is None:
        return None
    if hasattr(usage, "model_dump"):
        return usage.model_dump()
    return {
        "prompt_tokens": getattr(usage, "prompt_tokens", None),
        "completion_tokens": getattr(usage, "completion_tokens", None),
        "total_tokens": getattr(usage, "total_tokens", None),
    }

def mcp_result_to_text(result):
    return "\n".join(
        item.text for item in result.content if getattr(item, "type", None) == "text"
    )

THINK_RE = re.compile(r"<think>(.*?)</think>", re.DOTALL)

def split_thinking(content):
    """Extract <think>...</think> blocks (NIM/Gemma). Returns (reasoning, visible_content)."""
    if not content:
        return "", content or ""
    reasoning = "\n".join(m.strip() for m in THINK_RE.findall(content))
    visible = THINK_RE.sub("", content).strip()
    return reasoning, visible

def extract_reasoning(msg, response):
    """Pull reasoning from message.reasoning_content / message.reasoning / response.reasoning if present."""
    r = getattr(msg, "reasoning_content", None) or getattr(msg, "reasoning", None)
    if r is None:
        r = getattr(response, "reasoning", None)
    if r and hasattr(r, "model_dump"):
        r = r.model_dump()
    return r

# ── Observability helpers ────────────────────────────────────────────

def compute_prompt_hash(text):
    """First 8 hex chars of sha256 of the system prompt — stable ID for prompt variants."""
    return hashlib.sha256(text.encode()).hexdigest()[:8]

def compute_tools_hash(tool_schemas):
    """sha256 of the JSON-serialized MCP tool schema list — detects server drift."""
    return hashlib.sha256(json.dumps(tool_schemas, sort_keys=True).encode()).hexdigest()[:8]

def get_git_commit():
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return "unknown"

## Discover tools from the MCP server

Connects to the live server and converts each `@mcp.tool()` into the OpenAI
Chat Completions tool schema. The LLM's view of the tools is generated from the
same decorators that define the server, so there is no hand-written schema to drift.

In [5]:
async def load_tools_from_mcp():
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            listed = await session.list_tools()
    return [
        {
            "type": "function",
            "function": {
                "name": t.name,
                "description": t.description or "",
                "parameters": t.inputSchema,
            },
        }
        for t in listed.tools
    ]

tools = await load_tools_from_mcp()
print(f"Loaded {len(tools)} tools from MCP:")
for t in tools:
    print("  -", t["function"]["name"])

Loaded 10 tools from MCP:
  - list_outputs
  - get_task_output
  - save_evaluation
  - word_iou
  - null_accuracy
  - levenshtein
  - char_f1
  - set_f1
  - sequence_lcs
  - set_inclusion


In [6]:
tools_hash = compute_tools_hash(tools)
git_commit = get_git_commit()
print(f"tools_hash: {tools_hash}  |  git_commit: {git_commit}")

tools_hash: 7540a7e5  |  git_commit: 4d39558


## MCP tool call

In [7]:
@weave.op()
async def call_mcp_tool(tool_name, arguments, verbose=True):
    if verbose:
        log("MCP CALL", f"{tool_name}({pretty(arguments)})")
    t0 = time.perf_counter()
    async with streamablehttp_client(MCP_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            result = await session.call_tool(tool_name, arguments)
            text = mcp_result_to_text(result)
    dt = time.perf_counter() - t0
    if verbose:
        log(
            "MCP RESULT",
            f"isError={getattr(result, 'isError', None)}  elapsed={dt:.2f}s  chars={len(text)}\n{preview_text(text)}",
        )
    return text

In [8]:
@weave.op()
def llm_step(messages):
    """Traced LLM completion call.

    Explicit @weave.op() wrapper ensures NIM / custom-base_url calls appear in
    Weave even if the OpenAI auto-patch doesn't fire for non-default base_url clients.
    """
    return llm_client.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,
        tool_choice="auto",
        **COMPLETION_KWARGS,
    )


@weave.op()
def log_parse_failure(tool_name, raw_args, error_msg):
    """Emits a traced event for JSON argument-parse failures.

    Behavior is unchanged (args fall back to {}); this op makes the failure
    visible in Weave rather than only in the notebook print output.
    """
    return {"tool_name": tool_name, "raw_args": raw_args, "error": error_msg}

## System prompt

This is where the "analyze the output type" reasoning is defined. The agent is
given the predicted/expected values but NOT the declared metric — it must classify
each field and pick the tool itself.

In [9]:
# need a context specific input to the beginning + laying out issue.

In [10]:
METRIC_EVAL_SYSTEM = """
You are a careful evaluation assistant with access to metric-calculation tools over MCP.

Your job: evaluate ONE model output against its ground truth, choosing the right
metric for each field by REASONING about the data — the correct metric is never given to you.

Procedure:
1. Call get_task_output(task_id, run_id) to fetch the fields. Each field has a
   `predicted` value (model output) and an `expected` value (ground truth).
2. For EACH field, look at the actual values and classify the data shape:
     - Free-form / multi-word OCR text (a raw line as printed)      -> word_iou
     - A single short extracted value that may be absent (null/"")   -> null_accuracy,
       and ALSO levenshtein or char_f1 to score the content when present
     - A list / collection of items                                  -> set_f1 if order
       does not matter; sequence_lcs if order matters; set_inclusion if a ground-truth
       item may be embedded inside a longer predicted string
3. Call the chosen metric tool(s) for that field with its predicted and expected values.
4. Build a field_evaluations list. One entry per field:
     {"field": <field name>, "metric": <metric tool name you used>,
      "scores": <the dict the metric tool returned>,
      "rationale": <one short sentence on why that metric fits the data>}
   If you used more than one metric for a field, set "metric" to the primary one and
   include all returned numbers under "scores".
5. Call save_evaluation with task_id, benchmark_id, model_id, run_id, image_id (all
   exactly as returned by get_task_output) and your field_evaluations list.
6. Finish with a 1-2 line summary: which metric you picked per field and why.

Rules:
- Decide the metric ONLY from the predicted/expected values and the field name.
- Do not invent scores; always use the numbers the metric tools return.
- If a tool returns an "error" key, read it, fix your arguments, and retry once.
"""

## Agent loop

In [11]:
@weave.op()
async def run_agent(user_prompt, system_prompt, max_steps=MAX_STEPS, verbose=True,
                    backend=BACKEND, model=MODEL, task_id=None, run_id=None):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    total_usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    tool_calls_by_name = {}
    tool_time_by_name = {}
    tool_errors_by_name = {}
    steps_detail = []
    llm_time_total = 0.0
    wall_t0 = time.perf_counter()
    steps_run = 0
    stopped_reason = "max_steps"
    answer = "Stopped after maximum tool-calling steps."

    if verbose:
        log("AGENT START", f"Model: {model}  Backend: {backend}")

    for step in range(1, max_steps + 1):
        steps_run = step
        if verbose:
            log(f"LLM CALL {step}", f"{len(messages)} messages")

        t0 = time.perf_counter()
        try:
            response = llm_step(messages)
        except Exception as e:
            stopped_reason = "error"
            answer = f"LLM error: {e}"
            if verbose:
                log("LLM ERROR", str(e))
            break
        dt = time.perf_counter() - t0
        llm_time_total += dt

        usage = usage_to_dict(response)
        if usage:
            for k in total_usage:
                v = usage.get(k)
                if isinstance(v, int):
                    total_usage[k] += v

        choice = response.choices[0]
        msg = choice.message
        messages.append(msg)

        raw_content = getattr(msg, "content", None)
        thinking_nim, visible = split_thinking(raw_content)
        thinking_api = extract_reasoning(msg, response)

        # Capture per-step reasoning and metadata for Weave trace
        steps_detail.append({
            "step": step,
            "finish_reason": choice.finish_reason,
            "thinking_nim": thinking_nim or None,
            "thinking_api": str(thinking_api) if thinking_api else None,
            "tool_calls": [tc.function.name for tc in (msg.tool_calls or [])],
            "llm_time": dt,
            "usage": usage,
        })

        if verbose:
            if thinking_nim:
                log(f"REASONING (<think>) step {step}", preview_text(thinking_nim, 2000))
            if thinking_api:
                log(f"REASONING (api) step {step}", preview_text(str(thinking_api), 2000))
            log(
                f"LLM RESPONSE {step}",
                f"finish={choice.finish_reason}  elapsed={dt:.2f}s\n{preview_text(visible)}",
            )

        if not msg.tool_calls:
            stopped_reason = "answered"
            answer = msg.content
            if verbose:
                log("AGENT DONE", pretty(total_usage))
            break

        for tc in msg.tool_calls:
            name = tc.function.name
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError as e:
                log("ARG PARSE ERROR", f"{e}\n{tc.function.arguments}")
                # BUG (logged, not fixed): args silently fall back to {} — tool is called anyway.
                log_parse_failure(name, tc.function.arguments, str(e))
                args = {}
            tool_calls_by_name[name] = tool_calls_by_name.get(name, 0) + 1
            t_tool = time.perf_counter()
            tool_result = await call_mcp_tool(name, args, verbose=verbose)
            tool_time_by_name[name] = tool_time_by_name.get(name, 0.0) + (time.perf_counter() - t_tool)
            if '"error"' in tool_result or "isError=True" in tool_result:
                tool_errors_by_name[name] = tool_errors_by_name.get(name, 0) + 1
            messages.append(
                {
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "name": name,
                    "content": tool_result,
                }
            )
    else:
        if verbose:
            log("AGENT STOPPED", f"Hit max_steps={max_steps}")

    return {
        "answer": answer,
        "usage": total_usage,
        "messages": messages,
        "steps": steps_run,
        "stopped_reason": stopped_reason,
        "steps_detail": steps_detail,
        "tool_calls_by_name": tool_calls_by_name,
        "tool_time_by_name": tool_time_by_name,
        "tool_errors_by_name": tool_errors_by_name,
        "llm_time_total": llm_time_total,
        "wall_time_total": time.perf_counter() - wall_t0,
    }

In [12]:
def print_run_summary(result):
    u = result.get("usage", {})
    calls = result.get("tool_calls_by_name", {})
    times = result.get("tool_time_by_name", {})
    errs = result.get("tool_errors_by_name", {})
    print("\n" + "=" * 60)
    print("RUN SUMMARY")
    print(f"  stopped:    {result.get('stopped_reason')}")
    print(f"  steps:      {result.get('steps')}")
    print(f"  wall time:  {result.get('wall_time_total', 0):.2f}s  (LLM {result.get('llm_time_total', 0):.2f}s)")
    print(f"  tokens:     prompt={u.get('prompt_tokens', 0)}  completion={u.get('completion_tokens', 0)}  total={u.get('total_tokens', 0)}")
    if calls:
        print("  tool calls:")
        for name in sorted(calls):
            print(f"    {name:20s} n={calls[name]:3d}  time={times.get(name, 0):6.2f}s  errors={errs.get(name, 0)}")
    else:
        print("  tool calls: (none)")
    print("=" * 60)

In [13]:
# ── Phase 2: Post-run integrity checks ──────────────────────────────

METRIC_TOOLS = {
    "word_iou", "null_accuracy", "levenshtein",
    "char_f1", "set_f1", "sequence_lcs", "set_inclusion",
}


def extract_saved_evaluation(messages):
    """Return (list_of_save_args, call_count) by scanning assistant tool_calls."""
    saves = []
    for msg in messages:
        for tc in getattr(msg, "tool_calls", None) or []:
            if tc.function.name == "save_evaluation":
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except json.JSONDecodeError:
                    args = {}
                saves.append(args)
    return saves, len(saves)


def check_score_consistency(messages):
    """Verify every numeric score in save_evaluation appeared in a prior metric-tool result.

    This catches the worst silent failure: the agent hallucinating scores instead of
    forwarding what the metric tools actually returned.
    """
    observed = set()
    for msg in messages:
        if isinstance(msg, dict) and msg.get("role") == "tool" and msg.get("name") in METRIC_TOOLS:
            try:
                data = json.loads(msg["content"])
                for v in data.values():
                    if isinstance(v, (int, float)):
                        observed.add(float(v))
            except (json.JSONDecodeError, AttributeError, TypeError):
                pass

    saves, count = extract_saved_evaluation(messages)
    if not saves:
        return {"consistent": None, "missing": [], "note": "no save_evaluation found"}

    missing = []
    for fe in saves[0].get("field_evaluations", []):
        for key, val in fe.get("scores", {}).items():
            if isinstance(val, (int, float)):
                if not any(abs(float(val) - v) < 1e-9 for v in observed):
                    missing.append({"field": fe.get("field"), "score_key": key, "value": val})
    return {"consistent": len(missing) == 0, "missing": missing}


def check_retry_rule(messages):
    """For each tool error, check whether the same tool was re-called with different args."""
    call_map = {}  # tool_call_id -> (name, args)
    for msg in messages:
        for tc in getattr(msg, "tool_calls", None) or []:
            try:
                args = json.loads(tc.function.arguments or "{}")
            except json.JSONDecodeError:
                args = {}
            call_map[tc.id] = (tc.function.name, args)

    error_names = []
    for msg in messages:
        if isinstance(msg, dict) and msg.get("role") == "tool":
            content = msg.get("content", "")
            if '"error"' in content or "isError=True" in content:
                tc_id = msg.get("tool_call_id", "")
                if tc_id in call_map:
                    error_names.append(call_map[tc_id][0])

    retry_info = {}
    for name in set(error_names):
        total_calls = sum(1 for n, _ in call_map.values() if n == name)
        error_count = error_names.count(name)
        retry_info[name] = {
            "errors": error_count,
            "total_calls": total_calls,
            "retried": total_calls > error_count,
        }
    return {"tool_errors_with_retry": retry_info, "total_errors": len(error_names)}


@weave.op()
def run_integrity_report(result):
    """Run all Phase-2 checks, print a summary, and return a dict (traced by Weave)."""
    messages = result.get("messages", [])
    saves, save_count = extract_saved_evaluation(messages)
    consistency = check_score_consistency(messages)
    retries = check_retry_rule(messages)
    save_success = save_count == 1 and result.get("stopped_reason") == "answered"

    print("\n--- INTEGRITY REPORT ---")
    print(f"  save_evaluation called : {save_count}x  (save_success={save_success})")
    print(f"  score consistency      : consistent={consistency['consistent']}  missing={consistency.get('missing', [])}")
    if retries["total_errors"]:
        print(f"  retry behavior         : {retries['tool_errors_with_retry']}")
    else:
        print(f"  retry behavior         : no tool errors")
    print("------------------------")

    return {
        "save_count": save_count,
        "save_success": save_success,
        "score_consistency": consistency,
        "retry_behavior": retries,
    }

## Run one output

In [49]:
# Discover a few outputs to evaluate, then run the agent on the first one.
listing = json.loads(await call_mcp_tool("list_outputs", {"benchmark_id": "5", "limit": 3}, verbose=False))
print(pretty(listing))

item = listing["outputs"][0]
prompt = (
    f"Evaluate the model output with task_id={item['task_id']} and run_id={item['run_id']}. "
    f"Fetch it, choose the right metric for each field, compute the scores, and save the evaluation."
)

with weave.attributes({
    "prompt_hash": compute_prompt_hash(METRIC_EVAL_SYSTEM),
    "prompt_name": PROMPT_NAME,
    "tools_hash": tools_hash,
    "git_commit": git_commit,
    "completion_kwargs": json.dumps(COMPLETION_KWARGS, default=str),
    "mcp_url": MCP_URL,
}):
    result = await run_agent(
        prompt, METRIC_EVAL_SYSTEM,
        backend=BACKEND, model=MODEL,
        task_id=item["task_id"], run_id=item["run_id"],
        verbose=True,
    )
    report = run_integrity_report(result)

print_run_summary(result)
print("\nFINAL:\n", result["answer"])

weave: 🍩 https://wandb.ai/darc/metric-eval-agent/r/call/019eb2e3-ec75-7fb4-8b99-66e86ab2ac99


{
  "count": 3,
  "outputs": [
    {
      "task_id": "2450",
      "run_id": 0,
      "benchmark_id": "5",
      "model_id": "1",
      "model_name": "gpt-4",
      "image_id": "0"
    },
    {
      "task_id": "2450",
      "run_id": 1,
      "benchmark_id": "5",
      "model_id": "1",
      "model_name": "gpt-4",
      "image_id": "0"
    },
    {
      "task_id": "2450",
      "run_id": 2,
      "benchmark_id": "5",
      "model_id": "1",
      "model_name": "gpt-4",
      "image_id": "0"
    }
  ]
}

[11:55:42] AGENT START
Model: google/gemma-4-31b-it  Backend: nim

[11:55:42] LLM CALL 1
2 messages

[11:55:48] REASONING (api) step 1
The user wants me to evaluate a model output for a specific `task_id` (2450) and `run_id` (0).

1.  **Fetch the data**: I need to call `get_task_output(task_id="2450", run_id=0)`.
2.  **Analyze the fields**: For each field returned, I'll look at the `predicted` and `expected` values to determine the appropriate metric.
    *   Free-form OCR text -> `wo

weave: 🍩 https://wandb.ai/darc/metric-eval-agent/r/call/019eb2e3-ed2d-77ee-9c25-2fce2572d463



[11:56:04] LLM RESPONSE 5
finish=stop  elapsed=1.45s
I evaluated the field `first_channel` using `null_accuracy` to check for presence and `levenshtein` to score the content, as it is a single short extracted value.

[11:56:04] AGENT DONE
{
  "prompt_tokens": 11293,
  "completion_tokens": 927,
  "total_tokens": 12220
}

--- INTEGRITY REPORT ---
  save_evaluation called : 1x  (save_success=True)
  score consistency      : consistent=True  missing=[]
  retry behavior         : no tool errors
------------------------

RUN SUMMARY
  stopped:    answered
  steps:      5
  wall time:  21.65s  (LLM 20.83s)
  tokens:     prompt=11293  completion=927  total=12220
  tool calls:
    get_task_output      n=  1  time=  0.49s  errors=0
    levenshtein          n=  1  time=  0.07s  errors=0
    null_accuracy        n=  1  time=  0.07s  errors=0
    save_evaluation      n=  1  time=  0.18s  errors=0

FINAL:
 I evaluated the field `first_channel` using `null_accuracy` to check for presence and `levens

weave: 🍩 https://wandb.ai/darc/metric-eval-agent/r/call/019eb2e4-4280-7a2b-8fb5-84288868080b


## Optional: batch over multiple outputs

In [ ]:
async def run_eval_batch(items, system_prompt=METRIC_EVAL_SYSTEM, prompt_name=PROMPT_NAME, max_steps=MAX_STEPS, verbose=False):
    out = []
    prompt_hash = compute_prompt_hash(system_prompt)
    for it in items:
        prompt = (
            f"Evaluate the model output with task_id={it['task_id']} and run_id={it['run_id']}. "
            f"Fetch it, choose the right metric for each field, compute the scores, and save the evaluation."
        )
        with weave.attributes({
            "prompt_hash": prompt_hash,
            "prompt_name": prompt_name,
            "tools_hash": tools_hash,
            "git_commit": git_commit,
            "completion_kwargs": json.dumps(COMPLETION_KWARGS, default=str),
            "mcp_url": MCP_URL,
        }):
            res = await run_agent(
                prompt, system_prompt,
                backend=BACKEND, model=MODEL,
                task_id=it["task_id"], run_id=it["run_id"],
                max_steps=max_steps, verbose=verbose,
            )
        res["task_id"] = it["task_id"]
        res["run_id"] = it["run_id"]
        out.append(res)
        calls = res.get("tool_calls_by_name", {})
        metric_calls = sum(n for k, n in calls.items()
                           if k not in ("list_outputs", "get_task_output", "save_evaluation"))
        print(f"[task {it['task_id']} run {it['run_id']}] steps={res['steps']} "
              f"metric_calls={metric_calls} tokens={res['usage']['total_tokens']} "
              f"wall={res['wall_time_total']:.1f}s stopped={res['stopped_reason']}")
    return out

# results = await run_eval_batch(listing["outputs"])

## Phase 3: Frozen Dataset & Scorers

Publish a versioned Weave dataset from benchmarks 5, 6, 7, 10, 11, define four
scorers, and wire into `weave.Evaluation` so every prompt variant is scored on the
same fixed set of tasks.

**One-time setup:** run `python3 generate_gold_metrics.py` from the `mcp/` directory
to generate `gold_metrics.csv` before running `selection_accuracy_scorer`.

In [14]:
# Fetch outputs from all target benchmarks and publish as a versioned Weave dataset.
# Requires the MCP server to be running. Re-run only to refresh the frozen set.

EVAL_BENCHMARK_IDS = ["5", "6", "7", "10", "11"]
LIMIT_PER_BENCHMARK = 5

all_rows = []
for bid in EVAL_BENCHMARK_IDS:
    result_text = await call_mcp_tool(
        "list_outputs", {"benchmark_id": bid, "limit": LIMIT_PER_BENCHMARK}, verbose=False
    )
    listing = json.loads(result_text)
    outputs = listing.get("outputs", [])
    for it in outputs:
        all_rows.append({
            "task_id": it["task_id"],
            "run_id": it["run_id"],
            "benchmark_id": it["benchmark_id"],
        })
    print(f"  benchmark {bid}: {len(outputs)} outputs")

print(f"\nTotal: {len(all_rows)} rows across benchmarks {EVAL_BENCHMARK_IDS}")
dataset = weave.Dataset(name="metric-eval-frozen-v1", rows=all_rows)
dataset_ref = weave.publish(dataset)
print(f"Published: {dataset_ref.uri()}")

weave: 🍩 https://wandb.ai/darc/metric-eval-agent/r/call/019eb37d-08ba-7773-829a-fd74913a9a74


  benchmark 5: 5 outputs


weave: 🍩 https://wandb.ai/darc/metric-eval-agent/r/call/019eb37d-0c91-7f19-9a24-fca011b461cf
weave: 🍩 https://wandb.ai/darc/metric-eval-agent/r/call/019eb37d-0def-7f98-ac24-492507321b9f


  benchmark 6: 5 outputs
  benchmark 7: 5 outputs


weave: 🍩 https://wandb.ai/darc/metric-eval-agent/r/call/019eb37d-0e89-7bcd-adcc-4c43521915ad
weave: 🍩 https://wandb.ai/darc/metric-eval-agent/r/call/019eb37d-0f2d-7cbd-b824-f7ecde749f4c


  benchmark 10: 5 outputs
  benchmark 11: 5 outputs

Total: 25 rows across benchmarks ['5', '6', '7', '10', '11']


weave: 📦 Published to https://wandb.ai/darc/metric-eval-agent/weave/objects/metric-eval-frozen-v1/versions/YI1bjiRvCDhLxHUrkrbfX4RzqRx2dvlGYzsq2ABkpHw


Published: weave:///darc/metric-eval-agent/object/metric-eval-frozen-v1:YI1bjiRvCDhLxHUrkrbfX4RzqRx2dvlGYzsq2ABkpHw


# instead of uses task_id we can have a table that stores what it means -> we can put the image in W&B

In [38]:
print(len(all_rows))
for r in all_rows:
    print(r)

25
{'task_id': '2450', 'run_id': 0, 'benchmark_id': '5'}
{'task_id': '2450', 'run_id': 1, 'benchmark_id': '5'}
{'task_id': '2450', 'run_id': 2, 'benchmark_id': '5'}
{'task_id': '2451', 'run_id': 0, 'benchmark_id': '5'}
{'task_id': '2451', 'run_id': 1, 'benchmark_id': '5'}
{'task_id': '3780', 'run_id': None, 'benchmark_id': '6'}
{'task_id': '3781', 'run_id': None, 'benchmark_id': '6'}
{'task_id': '3783', 'run_id': None, 'benchmark_id': '6'}
{'task_id': '3789', 'run_id': None, 'benchmark_id': '6'}
{'task_id': '3785', 'run_id': None, 'benchmark_id': '6'}
{'task_id': '4018', 'run_id': None, 'benchmark_id': '7'}
{'task_id': '4004', 'run_id': None, 'benchmark_id': '7'}
{'task_id': '4022', 'run_id': None, 'benchmark_id': '7'}
{'task_id': '4020', 'run_id': None, 'benchmark_id': '7'}
{'task_id': '4006', 'run_id': None, 'benchmark_id': '7'}
{'task_id': '4435', 'run_id': None, 'benchmark_id': '10'}
{'task_id': '4436', 'run_id': None, 'benchmark_id': '10'}
{'task_id': '4437', 'run_id': None, 'benc

# can we pull trace information and put it in a table -> it's a bit hard to read right now

In [ ]:
# ── Scorers ──────────────────────────────────────────────────────────
# Each scorer receives `output` (the dict returned by run_agent) plus the
# dataset row fields as keyword arguments.

GOLD_CSV = Path("gold_metrics.csv")

def _load_gold_metrics():
    """Load gold metrics keyed by (benchmark_id, field_name). Returns {} if CSV missing."""
    if not GOLD_CSV.exists():
        print(f"Warning: {GOLD_CSV} not found — run `python3 generate_gold_metrics.py` first.")
        return {}
    gold = {}
    with open(GOLD_CSV) as f:
        for row in csv.DictReader(f):
            gold[(row["benchmark_id"], row["field_name"])] = row["gold_metric"]
    return gold


@weave.op()
def save_success_scorer(output):
    _, save_count = extract_saved_evaluation(output.get("messages", []))
    return {
        # >= 1 rather than == 1: a row that hit a validation error on the first
        # save attempt and retried successfully ends up with save_count=2, which
        # is still a success — only 0 saves is a genuine failure.
        "save_success": save_count >= 1 and output.get("stopped_reason") == "answered",
        "save_count": save_count,
    }


@weave.op()
def score_consistency_scorer(output):
    return check_score_consistency(output.get("messages", []))


@weave.op()
def efficiency_scorer(output):
    return {
        "steps": output.get("steps", 0),
        "tool_errors": sum(output.get("tool_errors_by_name", {}).values()),
        "total_tokens": output.get("usage", {}).get("total_tokens", 0),
    }


@weave.op()
def selection_accuracy_scorer(output, benchmark_id=None, **kwargs):
    """Fraction of fields where the agent chose the correct primary metric.

    Returns None until gold_metrics.csv exists (run generate_gold_metrics.py first).
    Gold metric = first entry in the benchmark's ground_truth metrics list.
    """
    gold = _load_gold_metrics()
    if not gold or benchmark_id is None:
        return None

    saves, _ = extract_saved_evaluation(output.get("messages", []))
    if not saves:
        return None

    field_evals = saves[0].get("field_evaluations", [])
    scoreable = [fe for fe in field_evals if (str(benchmark_id), fe.get("field")) in gold]
    if not scoreable:
        return None

    correct = sum(
        1 for fe in scoreable
        if fe.get("metric") == gold[(str(benchmark_id), fe.get("field"))]
    )
    return {
        "selection_accuracy": correct / len(scoreable),
        "correct": correct,
        "total": len(scoreable),
    }


print("Scorers ready: save_success | score_consistency | efficiency | selection_accuracy")

In [ ]:
import asyncio

# Cap concurrent agent runs. weave.Evaluation runs workers in parallel by default;
# NIM and the MCP server both drop connections under heavy load, so limit to 2.
_eval_semaphore = asyncio.Semaphore(2)

@weave.op()
async def model_fn(task_id, run_id, benchmark_id, **kwargs):
    """Weave Evaluation model: runs the agent on one dataset row."""
    safe_run_id = int(run_id) if run_id is not None else 0
    # Use safe_run_id in the prompt so the agent never sees "run_id=None"
    # and doesn't pass null to save_evaluation (which requires an int).
    prompt = (
        f"Evaluate the model output with task_id={task_id} and run_id={safe_run_id}. "
        f"Fetch it, choose the right metric for each field, compute the scores, and save the evaluation."
    )
    async with _eval_semaphore:
        return await run_agent(
            prompt, METRIC_EVAL_SYSTEM,
            backend=BACKEND, model=MODEL,
            task_id=task_id, run_id=safe_run_id,
        )

evaluation = weave.Evaluation(
    name=f"metric-eval-{PROMPT_NAME}",
    dataset=dataset,
    scorers=[save_success_scorer, score_consistency_scorer, efficiency_scorer, selection_accuracy_scorer],
)
print(f"Evaluation '{evaluation.name}' ready — {len(all_rows)} rows, 4 scorers.")
print("Run the cell below to execute.")

- you may need to do a specific parallel call for W&B, look into that
- we need to calc tokens/second within script
- reasoning (how much reasoning effort and what modl we used -> all of the metadata)
- context window size
- RAM usage
- GPU usage (SM utilization), does this we change it if we're doing individual vs. paralleling (key question)

- We may not need to focus on the research tool call choice (tool accuracy) vs. the ease of seeing which tools it called (observability)

- We can also make a dashboard to track agent actions -> can we use W&B? or should we make our own?
- Centralizing data: we could store in Yen/Redivis/MongoDB

Main question:

- We need to change add what we're tracking
- Figure out how to call in batch vs. parallel
- How to visualize

In [17]:
# Runs run_agent for every row in the dataset — expects ~30s per row on NIM.
# Each row also saves results to MongoDB via save_evaluation.

eval_results = await evaluation.evaluate(model_fn)


[14:43:09] AGENT START
Model: google/gemma-4-31b-it  Backend: nim

[14:43:09] LLM CALL 1
2 messages


weave: 🍩 https://wandb.ai/darc/metric-eval-agent/r/call/019eb37d-3d13-7d2c-84b9-bbed2b06d64f



[14:43:16] REASONING (api) step 1
The user wants me to evaluate a model output for a specific `task_id` (2450) and `run_id` (0).

1.  **Fetch the data**: Call `get_task_output(task_id="2450", run_id=0)`.
2.  **Analyze fields**: For each field in the output, I need to look at the `predicted` and `expected` values to determine the appropriate metric.
    *   Free-form OCR text -> `word_iou`
    *   Single short extracted value (possibly null) -> `null_accuracy` + (`levenshtein` or `char_f1`)
    *   List/Collection (order doesn't matter) -> `set_f1`
    *   List/Collection (order matters) -> `sequence_lcs`
    *   List/Collection (substring match) -> `set_inclusion`
3.  **Calculate scores**: Call the chosen metric tools.
4.  **Save evaluation**: Call `save_evaluation` with the results.
5.  **Summarize**: Provide a brief summary of the metrics used.

[14:43:16] LLM RESPONSE 1
finish=tool_calls  elapsed=6.53s


[14:43:16] MCP CALL
get_task_output({
  "run_id": 0,
  "task_id": "2450"
})

[

Error in post_writer
Traceback (most recent call last):
  File "/home/users/ltdarc/zfs/projects/students/ltdarc-usf-intern-2025/venv/lib/python3.10/site-packages/httpx/_transports/default.py", line 101, in map_httpcore_exceptions
    yield
  File "/home/users/ltdarc/zfs/projects/students/ltdarc-usf-intern-2025/venv/lib/python3.10/site-packages/httpx/_transports/default.py", line 394, in handle_async_request
    resp = await self._pool.handle_async_request(req)
  File "/home/users/ltdarc/zfs/projects/students/ltdarc-usf-intern-2025/venv/lib/python3.10/site-packages/httpcore/_async/connection_pool.py", line 256, in handle_async_request
    raise exc from None
  File "/home/users/ltdarc/zfs/projects/students/ltdarc-usf-intern-2025/venv/lib/python3.10/site-packages/httpcore/_async/connection_pool.py", line 236, in handle_async_request
    response = await connection.handle_async_request(
  File "/home/users/ltdarc/zfs/projects/students/ltdarc-usf-intern-2025/venv/lib/python3.10/site-packag


[14:50:08] REASONING (api) step 3
The field `first_program_trailing` has a predicted value of `"(CC)"` and an expected value of `[]`.
The expected value is a list (empty list), and the predicted value is a string `"(CC)"`.
Wait, the predicted value is a string, but the expected value is a list.
Looking at the `set_f1` tool call, I passed `predicted=["(CC)"]` (implicitly or explicitly as a list) and `expected=[]`.
Actually, the `predicted` value in the `get_task_output` was `"(CC)"` (a string).
The `expected` value was `[]` (a list).
Usually, if the expected is a list, the predicted should be a list.
If the model output a string instead of a list, it's a format error, but `set_f1` handles lists.
Let's re-examine the data:
`predicted`: `"(CC)"`
`expected`: `[]`
This looks like a list of trailing markers/tags.
Since the expected is a list, `set_f1` is appropriate for a collection of items.
The `set_f1` result was 0.0.

Wait, let me double check if `predicted` was a string or a list.
`"pr

...

weave: Evaluated 22 of 25 examples
weave: Evaluated 23 of 25 examples



[15:01:36] MCP RESULT
isError=False  elapsed=27.63s  chars=177
{
  "saved": true,
  "collection": "agentic_evaluations",
  "key": {
    "task_id": "4018",
    "benchmark_id": "7",
    "model_id": "10",
    "run_id": 0
  },
  "n_fields": 3
}

[15:01:36] LLM CALL 9
18 messages

[15:01:46] REASONING (api) step 9
The user wants me to evaluate a model output for `task_id=4018`.
I have already:
1. Fetched the task output using `get_task_output`.
2. Analyzed the fields:
    - `first_channel_raw`: Predicted "2 KCTV CBS", Expected "4 WDAF". This is free-form OCR text, so I used `word_iou`.
    - `first_channel_name`: Predicted "KCTV CBS", Expected "WDAF". This is a single short extracted value, so I used `null_accuracy` and `levenshtein`.
    - `first_channel_numbers`: Predicted ["2"], Expected ["4"]. This is a list of items, so I used `set_f1`.
3. Computed the scores using the respective tools.
4. Attempted to save the evaluation. The first attempt failed because `run_id` was `null` (None), a

weave: Evaluated 24 of 25 examples



[15:02:16] REASONING (api) step 9
The user wants me to evaluate a model output.
I already fetched the output for `task_id="4022"`.
The `run_id` returned by `get_task_output` was `null`.
When I tried to call `save_evaluation`, I passed `run_id="null"` (as a string) or `None`, and it failed because `run_id` must be an integer.
Wait, looking at the `get_task_output` response: `"run_id": null`.
Looking at the `list_outputs` response: `"run_id": null`.
The `save_evaluation` tool definition says `run_id` is an `INTEGER`.
If the `run_id` is `null` in the source, but the tool requires an `INTEGER`, there's a conflict.
However, usually, if a `run_id` is not provided or is null, it might be 0 or some default, but the error message says `Input should be a valid integer [type=int_type, input_value=None, input_type=NoneType]`.
Wait, I passed `run_id: "null"` in my previous `save_evaluation` call. That's a string.
If I pass `run_id: 0`, it might work, but I should check if there's a way to handle `

weave: Evaluated 25 of 25 examples
weave: Evaluation summary {
weave:   "output": {
weave:     "usage": {
weave:       "prompt_tokens": {
weave:         "mean": 18498.875
weave:       },
weave:       "completion_tokens": {
weave:         "mean": 3091.25
weave:       },
weave:       "total_tokens": {
weave:         "mean": 21590.125
weave:       }
weave:     },
weave:     "steps": {
weave:       "mean": 6.375
weave:     },
weave:     "tool_calls_by_name": {
weave:       "get_task_output": {
weave:         "mean": 1.0
weave:       },
weave:       "null_accuracy": {
weave:         "mean": 1.25
weave:       },
weave:       "levenshtein": {
weave:         "mean": 1.25
weave:       },
weave:       "save_evaluation": {
weave:         "mean": 1.75
weave:       },
weave:       "word_iou": {
weave:         "mean": 1.0
weave:       },
weave:       "set_f1": {
weave:         "mean": 1.0
weave:       },
weave:       "list_outputs": {
weave:         "mean": 1.0
weave:       },
weave:       "set_incl